In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("burak29/Natural_Language_to_Ffmpeg_Commands", split="train")

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zhv2usi0/unsloth_33b6f2b89f8b4f849467bcb0e6d7100b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zhv2usi0/unsloth_33b6f2b89f8b4f849467bcb0e6d7100b
  Resolved https://github.com/unslothai/unsloth.git to commit be4598e16cfc3c802c8d3664ff97e4d7e41844fc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

In [ ]:
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.1 MB/s eta 0:00:00


In [ ]:
from unsloth import FastLanguageModel

# This line defines both 'model' and 'tokenizer'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

In [ ]:
SYSTEM_PROMPT = """You are an expert video engineer. Translate the user's natural language request into a valid FFmpeg command. Output ONLY the raw FFmpeg command. Do not provide markdown formatting, explanations, or quotes."""

# 3. Create the Formatting Function
def formatting_prompts_func(examples):
    # Extract the two columns from the burak29 dataset
    natural_language_prompts = examples["natural_language"]
    ffmpeg_commands = examples["ffmpeg_command"]

    texts = []

    # Loop through the batch and build the ChatML structure
    for nl, ffmpeg in zip(natural_language_prompts, ffmpeg_commands):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": nl},
            {"role": "assistant", "content": ffmpeg}
        ]

        # apply_chat_template converts the JSON-like list into the exact string Qwen expects
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return { "text" : texts }

In [ ]:
# batched=True makes this run much faster
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

In [ ]:
print(formatted_dataset["text"][0])

<|im_start|>system
You are an expert video engineer. Translate the user's natural language request into a valid FFmpeg command. Output ONLY the raw FFmpeg command. Do not provide markdown formatting, explanations, or quotes.<|im_end|>
<|im_start|>user
resize input.jpg to 512x512<|im_end|>
<|im_start|>assistant
ffmpeg -i input.jpg -vf scale=512:512 output.jpg<|im_end|>



In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The adapter size. 16 is standard for syntax/coding tasks.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Target all linear layers
    lora_alpha = 16, # Scaling factor
    lora_dropout = 0, # Keep at 0 for optimized training
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial: saves huge amounts of VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 2. Set up the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset, # The dataset you mapped in the previous step
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Simulates a batch size of 8
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(), # Uses newer, faster math if your GPU supports it
        logging_steps = 10,
        optim = "adamw_8bit", # Critical for preventing memory crashes on free GPUs
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1030 [00:00<?, ? examples/s]

In [ ]:
# 3. Start the fine-tuning loop
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,030 | Num Epochs = 3 | Total steps = 387
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
10,0.448492
20,0.410852
30,0.420593
40,0.430091
50,0.411039
60,0.424453
70,0.452264
80,0.456008
90,0.450122
100,0.420799


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-387/tokenizer_config.json.


In [ ]:
# 4. Save the Model Locally
model.save_pretrained("qwen-ffmpeg-model")
tokenizer.save_pretrained("qwen-ffmpeg-model")

# Optional: Export directly to GGUF format for use in Ollama
# This takes a few minutes but creates a highly portable single file
model.save_pretrained_gguf("qwen-ffmpeg-model", tokenizer, quantization_method = "q4_k_m")

Unsloth: Restored added_tokens_decoder metadata in qwen-ffmpeg-model/tokenizer_config.json.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

... (16 outputs left out)

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:11<00:00, 62.94s/it]


Unsloth: Merge process complete. Saved to `/content/qwen-ffmpeg-model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11115-mix-a6922cc (app-b11115-mix-a6922cc-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen-ffmpeg-model_gguf/qwen2.5-coder-7b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen-ffmpeg-model_gguf/qwen2.5-coder-7b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model qwen-ffmpeg-model_gguf/qwen2.5-coder-7b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to qwen-ffmpeg-model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f qwen-ffmpeg-model_gguf/Modelfile


{'save_directory': 'qwen-ffmpeg-model',
 'gguf_directory': 'qwen-ffmpeg-model_gguf',
 'gguf_files': ['qwen-ffmpeg-model_gguf/qwen2.5-coder-7b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'qwen-ffmpeg-model_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
# The test request you want to convert
user_request = "Convert an MKV video to MP4 and scale it down to 720p."

# Construct the exact same prompt structure used in training
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_request}
]

# Format and tokenize the input
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Crucial: tells the model it's time to generate the assistant response
    return_tensors = "pt",
).to("cuda") # Move the input tensors to the GPU

In [ ]:
from transformers import TextStreamer

# 1. The test request you want to convert
user_request = "Convert an MKV video to MP4 and scale it down to 720p."

# 2. Construct the exact same prompt structure used in training
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_request}
]

# 3. Format into a string first, then tokenize
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

# 4. Stream the generation to the console live
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

outputs = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 250,
    use_cache = True,
    do_sample = False,
)

# 5. Capture the output into a Python string
# outputs[0] contains the original prompt + the generated tokens.
# We slice off the prompt by finding the length of the original input.
input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]

# Decode only the new tokens.
# skip_special_tokens=True automatically strips <|im_end|>
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

clean_command = generated_text.strip()

print("\n--- Saved to Variable ---")
print(clean_command)

Both `max_new_tokens` (=250) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ffmpeg -i input.mkv -vf 'scale=1280:720' -c:v libx264 -crf 23 -c:a aac output.mp4<|im_end|>

--- Saved to Variable ---
ffmpeg -i input.mkv -vf 'scale=1280:720' -c:v libx264 -crf 23 -c:a aac output.mp4


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Load the secret securely
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
from huggingface_hub import HfApi, login
from google.colab import userdata

# 2. Initialize the API
api = HfApi()

# 3. Upload the existing local folder directly
api.upload_folder(
    folder_path="qwen-ffmpeg-model_gguf",
    repo_id="mohammed-usmani/ffmpeg-model", # Replace with your username
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/mohammed-usmani/ffmpeg-model/commit/3ba72f634a3ce33e65d2d0dd2a3c1bb32611b1b2', commit_message='Upload folder using huggingface_hub', commit_description='', oid='3ba72f634a3ce33e65d2d0dd2a3c1bb32611b1b2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mohammed-usmani/ffmpeg-model', endpoint='https://huggingface.co', repo_type='model', repo_id='mohammed-usmani/ffmpeg-model'), pr_revision=None, pr_num=None)